# Importing Libraries

In [1]:
! pip install mycolorpy

  Preparing metadata (setup.py) ... done
  Created wheel for mycolorpy: filename=mycolorpy-1.5.1-py3-none-any.whl size=3873 sha256=83d1af753f144506f7ecb8d937083d6f4c485f8d4bf788a67fb2f47c98de88e5
  Stored in directory: /root/.cache/pip/wheels/e6/a2/70/8113826487ef774503bcd38963b04b4c920deef45d7d54993e
Successfully built mycolorpy


In [ ]:
####### Importing Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#plt.switch_backend('agg')
import tensorflow as tf
import os                                                                                                         
import gc
import math
import pydot
from scipy.spatial import distance
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
import scipy.special as sp
from sklearn.preprocessing import normalize as norm
import argparse

: 

# Loading Dataset

In [2]:
####### Loading Datafiles
#Train_Embeddings = np.load('/kaggle/input/dgbqa-vanilla-res3d-vivit-1-1-uns5-5-iar-soli/DGBQA_Vanilla_Res3D-ViViT_1-1_UNS5-5_IAR_SOLI_Train.npz',allow_pickle=True)['arr_0']
#Test_Embeddings = np.load('/kaggle/input/dgbqa-hgr-res3dvivit-tiny-embeddings/DGBQA_HGR_Res3D-ViViT_2-5-15_Tiny.npz',allow_pickle=True)['arr_0']
#y_train = np.load('/kaggle/input/rhga-deltadistance-soli-dataset/y_train_DeltaDistance_SOLI.npz',allow_pickle=True)['arr_0']
y_dev = np.load('./Embeddings/y_dev_DGBQA_Seen_HandLogin.npz',allow_pickle=True)['arr_0']
#y_train_id = np.load('/kaggle/input/rhga-deltadistance-soli-dataset/y_train_id_DeltaDistance_SOLI.npz',allow_pickle=True)['arr_0']
y_dev_id = np.load('./Embeddings/y_dev_id_DGBQA_Seen_HandLogin.npz',allow_pickle=True)['arr_0']

# Delta Distance

## Delta Distance

In [ ]:
####### Delta-Distance Score
###### Defining Essentials
num_gestures = 11 # Number of Gesture Classes
num_subjects = 10 # Number of Subjects
#num_gesture_per_g_s = 10 # Total Instances of a specific gesture per person

###### Delta-Distance Score
def delta_dist_compute(embeddings,g_id,num_subjects,y_dev,y_dev_id):

    """
    Function to Compute Delta Distance for a Gesture

    INPUTS:-
    1) embeddings: Feature Embeddings
    2) g_id: Index Value of the Gesture Class
    3) num_subjects: Number of Subjects
    4) y_dev: Ground-Truth Gesture Labels
    5) y_dev_id: Ground-Truth Subject Labels

    OUTPUTS:-
    1) delta_dist: |d_cs - d_c|/d_c, Delta Distance Computed for a particular gesture

    """
    ###### Defining Essentials
    d_cs = [] # List to Maximal Intra-Subject Distance

    ###### Iterating over Subjects
    for s_id in range(num_subjects): 

        ##### Defining Essentials
        embedding_store_s = [] # List to Store Embeddings from Gesture - 'g_id' and Subject - 's_id'
        dist_store_s = [] # List to Store Distance within subject 's_id' 

        ##### Curating Required Gesture List from g_id and s_id
        for idx in range(y_dev.shape[0]): # Iterating over Embeddings

            if(y_dev[idx] == g_id and y_dev_id[idx] == s_id):
                embedding_store_s.append(embeddings[idx]) # Storing the Required Embeddings
        
        ##### Computing Intra-Gesture and Intra-Subject Distances
        for emb_query_idx, emb_query in enumerate(embedding_store_s):

            if(emb_query_idx != (len(embedding_store_s)-1)): # Checking if the Current Query is the Last Query

                for emb_key_idx in range(emb_query_idx+1,len(embedding_store_s),1): # Iterating over the Embeddings

                    emb_key_curr = embedding_store_s[emb_key_idx] # Extracting Current Embedding Key
                    dist_curr = distance.euclidean(emb_query,emb_key_curr) # Current Distance 
                    dist_store_s.append(dist_curr) # Appending the Computed Distance to dist_curr

        ##### Computing Maximal Distance for the Current Gesture and Subject
        d_cs_curr = np.max(dist_store_s)
        d_cs.append(d_cs_curr) # Storing Values

    ###### Computing Delta Distance
    d_c = np.max(d_cs) # Maximal Distance Amongst all the Subjects
    print('Average Intra-Subject Separation: '+str(np.average(d_cs)))
    print('Minimum Intra-Subject Separation: '+str(np.min(d_cs)))
    print('Maximum Intra-Subject Separation (d_cs): '+str(d_c))
    delta_dist = np.average(np.abs(d_cs - d_c)/d_c) # Averaging Computed Delta Distance Per Subject
    
    return delta_dist

In [ ]:
####### Computing Delta Distance
###### Defining Essentials
gesture_list = ["PinchIndex", "PalmTilt", "FingerSlider", "PinchPinky", "SlowSwipeRL", "FastSwipeRL", "Push", "Pull", "FingerRub", "Circle", "PalmHold"]
num_subjects = 10

###### Iterating over Gestures
for g_id, gesture_curr in enumerate(gesture_list):
    print('==============================================')
    delta_dist_g = delta_dist_compute(Test_Embeddings,g_id,num_subjects,y_dev,y_dev_id)
    print('Delta Distance for '+str(gesture_curr)+' = '+str(delta_dist_g))                            

## GBQA - Delta Distance

In [3]:
####### GBQA - Delta Distance Score
def gbqa_delta_dist_compute(embeddings,g_id,num_subjects,y_dev,y_dev_id):

    """
    Function to Compute Delta Distance for a Gesture

    INPUTS:- 
    1) embeddings: Feature Embeddings
    2) g_id: Index Value of the Gesture Class
    3) num_subjects: Number of Subjects
    4) y_dev: Ground-Truth Gesture Labels
    5) y_dev_id: Ground-Truth Subject Labels

    OUTPUTS:-
    1) gbqa_delta_distance: d_c_star/d_cs, GBQA Delta Distance Computed for a particular gesture

    """
    ###### Defining Essentials
    d_cs = [] # List to Maximal Intra-Subject Distance
    emb_avg_s = [] # List to Store Subject Specific Gesture Centroids

    ###### Iterating over Subjects
    for s_id in range(num_subjects): 

        ##### Defining Essentials
        embedding_store_s = [] # List to Store Embeddings from Gesture - 'g_id' and Subject - 's_id'
        dist_store_s = [] # List to Store Distance within subject 's_id'

        ##### Intra-Subject Distance
        #### Curating Required Gesture List from g_id and s_id
        for idx in range(y_dev.shape[0]): # Iterating over Embeddings

            if(y_dev[idx] == g_id and y_dev_id[idx] == s_id):
                embedding_store_s.append(embeddings[idx]) # Storing the Required Embeddings

        #### Computing Intra-Gesture and Intra-Subject Distances
        for emb_query_idx, emb_query in enumerate(embedding_store_s):

            if(emb_query_idx != (len(embedding_store_s)-1)): # Checking if the Current Query is the Last Query

                for emb_key_idx in range(emb_query_idx+1,len(embedding_store_s),1): # Iterating over the Embeddings

                    emb_key_curr = embedding_store_s[emb_key_idx] # Extracting Current Embedding Key
                    dist_curr = distance.euclidean(emb_query,emb_key_curr) # Current Distance 
                    dist_store_s.append(dist_curr) # Appending the Computed Distance to dist_curr

        #### Computing Maximal Distance for the Current Gesture and Subject
        d_cs_curr = np.max(dist_store_s)
        d_cs.append(d_cs_curr) # Storing Values

        ##### Inter-Subject Distance
        #### Subject's Gesture Centroid
        emb_avg_s_curr = np.average(embedding_store_s,axis=0) # Subject Specific Gesture Centroid
        emb_avg_s.append(emb_avg_s_curr)

    ###### Computing Avg. Maximal Intra-Subject Distance
    d_cs_avg = np.average(d_cs)
    
    ###### Computing Inter-Subject Distance
    ##### Defining Essentials
    dist_inter = [] # List to store Inter-Subject Distances

    ##### Computing Distances amongst the Subject Centroids
    for emb_query_idx, emb_query in enumerate(emb_avg_s):

            if(emb_query_idx != (len(emb_avg_s)-1)): # Checking if the Current Query is the Last Query

                for emb_key_idx in range(emb_query_idx+1,len(emb_avg_s),1): # Iterating over the Embeddings

                    emb_key_curr = emb_avg_s[emb_key_idx] # Extracting Current Embedding Key
                    dist_curr = distance.euclidean(emb_query,emb_key_curr) # Current Distance 
                    dist_inter.append(dist_curr) # Appending the Computed Distance to dist_curr  

    ##### Computing Average Inter-Subject Distance
    d_c_star = np.average(dist_inter)

    ###### Computing GBQA Distance Delta Score
    print('Inter-Distance - d_c_star: '+str(d_c_star))
    print('Intra-Distance - d_cs_avg: '+str(d_cs_avg))
    #gbqa_delta_distance = math.exp(d_c_star - d_cs_avg)
    #gbqa_delta_distance = math.exp(d_c_star - d_cs_avg) + 1*((d_c_star/d_cs_avg))
    gbqa_delta_distance = math.exp(d_c_star - d_cs_avg) - (1*(d_cs_avg/d_c_star)) # For Seen Identities
    dgbqa_score_wo = math.exp(d_c_star - d_cs_avg)
    #gbqa_delta_distance = math.exp(d_c_star - d_cs_avg) - (0.2*(d_cs_avg/d_c_star)) # For Unseen Identities: UNS

    return gbqa_delta_distance, d_c_star, d_cs_avg, dgbqa_score_wo

In [4]:
###### Rank Derivation
def rank_compute(rank_vector,val_to_rank,reverse_flag):

    """
    Function to derive rank of a particular value

    INPUTS:-
    1) rank_vector: Vector in which ranking is to be performed
    2) val_to_rank: Value whoose rank is to be derived
    3) reverse_flag: Flag to signify if the order of sort is to be reversed

    OUTPUTS:-
    1) rank_val: Ranked Value - The best is rank '0'
    """ 

    if(reverse_flag == True):
        rank_vector_sort = -(np.sort(-rank_vector))
    else:
        rank_vector_sort = np.sort(rank_vector) # Sorting the Vector    
    for item_idx,item in enumerate(rank_vector_sort): # Iterating over the vector
        if(item == val_to_rank): # Match-Found
            rank_val = item_idx # Rank Assignment 
            break

    return rank_val

###### Acceptance Score
def compute_acceptance_score(eer_vec,dgbqa_vec,G_Total):
    
    """
    Computation of Acceptance Score
    
    INPUTS:-
    1) eer_vec: Vector of EER values
    2) dgbqa_vec: Vector of DGBQA values
    3) G_Total: Total gestures in the vector
    
    OUTPUTS:-
    1) acceptance_score: Acceptance Score values
    """
    ###### Iterating over Gestures
    acceptance_score = 0 # Current Value of Acceptance Score

    for g_idx in range(G_Total):
        
        #### Values of the Current Gesture
        eer_val_curr = eer_vec[g_idx] # EER Value
        dgbqa_val_curr = dgbqa_vec[g_idx] # DGBQA Value
        
        #### Rank of the Current Gesture
        eer_rank_curr = rank_compute(eer_vec,eer_val_curr,True) # EER Rank
        dgbqa_rank_curr = rank_compute(dgbqa_vec,dgbqa_val_curr,False) # DGBQA Rank
        
        #### Acceptance Score
        rank_penalty = 1/(1 + np.abs(eer_rank_curr - dgbqa_rank_curr))
        corr_score = (eer_val_curr*dgbqa_val_curr)
        acceptance_score = acceptance_score + (-rank_penalty*corr_score)
        
    return acceptance_score

In [5]:
###### Avg. Rank Deviation
def avg_rank_deviation(eer_vec,dgbqa_vec,G_Total):
    
    """
    Computation of Acceptance Score
    
    INPUTS:-
    1) eer_vec: Vector of EER values
    2) dgbqa_vec: Vector of DGBQA values
    3) G_Total: Total gestures in the vector
    
    OUTPUTS:-
    1) avg_rank_deviation: Total Deviation in Rank/Total Number of Gestures
    """
    
    rank_deviation = 0 # Total Rank Deviation
    
    for g_idx in range(G_Total):
        
        #### Values of the Current Gesture
        eer_val_curr = eer_vec[g_idx] # EER Value
        dgbqa_val_curr = dgbqa_vec[g_idx] # DGBQA Value
        
        #### Rank of the Current Gesture
        eer_rank_curr = rank_compute(eer_vec,eer_val_curr,True) # EER Rank
        dgbqa_rank_curr = rank_compute(dgbqa_vec,dgbqa_val_curr,False) # DGBQA Rank
        
        #### Rank Difference
        rank_deviation =  rank_deviation + np.abs(eer_rank_curr - dgbqa_rank_curr)
        
    avg_rank_deviation = rank_deviation/G_Total # Computing Avg. over all the gestures
    return avg_rank_deviation

In [14]:
###### Comparitive Bar Graphs
def comparitive_bars(eer_score, dgbqa_score, num_gestures, gesture_list, filepath):
    """
    Bar Graphs for Comparitive Analysis
    
    INPUTS:-
    1) eer_score: Normalized EER-Score Vector
    2) dgbqa_score: Normalized DGBQA-Score Vector
    3) num_gestures: Total Gestures
    4) gesture_list: Name of the Gestures
    5) filepath: Path to save the plot
    """
    
    ##### EER-Plot
    x_axes = np.arange(start=0,stop=4*2,step=2)
    plt.bar(x_axes,eer_score,label='EER-Score',color='gold')
    
    ##### DGBQA-Bar
    plt.bar(x_axes+0.8,-dgbqa_score,label='-(DGBQA-Score)',color='tab:red')
    
    ##### Graph Edits
    plt.xlabel(None)
    plt.xticks(x_axes+0.4,labels=gesture_list,fontsize=20,rotation=30)
    plt.yticks(fontsize=15)
    plt.tick_params(bottom=True,left=True)
    plt.legend(frameon=True,fontsize=15)
    plt.savefig(filepath)
    plt.show()

In [7]:
####### DGBQA Score
###### Defining Essentials
gesture_list = ['Compass','Piano','Push','Flipping Fist']
num_subjects = 16
num_gestures = 4
dgbqa_score = []
d_c_star = []
d_cs = []
dgbqa_score_wo = []
#eer_score = np.array([0.44,1.29,4.89,1.05])
#eer_score = eer_score/np.linalg.norm(eer_score)
Test_Embeddings = np.load('./Embeddings/DGBQA_CGID_Res3D-MF_1-1_HandLogin.npz',allow_pickle=True)['arr_0']

###### Iterating over Gestures
for g_id, gesture_curr in enumerate(gesture_list):
    print('==============================================')
    dgbqa_score_curr, d_c_star_curr, d_cs_curr, dgbqa_score_wo_curr = gbqa_delta_dist_compute(Test_Embeddings,g_id,num_subjects,y_dev,y_dev_id)
    dgbqa_score.append(dgbqa_score_curr)
    d_c_star.append(d_c_star_curr)
    d_cs.append(d_cs_curr)
    dgbqa_score_wo.append(dgbqa_score_wo_curr)
    #print('GBQA Delta Distance for '+str(gesture_curr)+' = '+str(dgbqa_score_curr))  

print(dgbqa_score)

dgbqa_score = np.array(dgbqa_score) # Array Formation
dgbqa_score = (dgbqa_score - np.mean(dgbqa_score))/np.std(dgbqa_score) # Mean Normalization
dgbqa_score = dgbqa_score/np.linalg.norm(dgbqa_score) # L2-Normalization

d_c_star = np.array(d_c_star) # Array Formation
d_c_star = (d_c_star - np.mean(d_c_star))/np.std(d_c_star) # Mean Normalization
d_c_star = d_c_star/np.linalg.norm(d_c_star) # L2-Normalization

d_cs = np.array(d_cs) # Array Formation
d_cs = (d_cs - np.mean(d_cs))/np.std(d_cs) # Mean Normalization
d_cs = d_cs/np.linalg.norm(d_cs) # L2-Normalization

dgbqa_score_wo = np.array(dgbqa_score_wo) # Array Formation
dgbqa_score_wo = (dgbqa_score_wo - np.mean(dgbqa_score_wo))/np.std(dgbqa_score_wo) # Mean Normalization
dgbqa_score_wo = dgbqa_score_wo/np.linalg.norm(dgbqa_score_wo) # L2-Normalization


###### Acceptance Score
#dgbqa_score_norm = dgbqa_score/np.linalg.norm(dgbqa_score)
#acceptance_score = -(np.sum(np.multiply(eer_score,dgbqa_score_norm)))
#ranked_acceptance_score = compute_acceptance_score(eer_score,dgbqa_score_norm,4)
#print('==============================================')
#print('Acceptance-Score: '+str(acceptance_score))
#print('Acceptance-Score-Norm: '+str(acceptance_score/num_gestures)) 
#print('Ranked Acceptance-Score: '+str(ranked_acceptance_score))

###### Comparitive Bar Graphs
#plt.rcParams["figure.figsize"] = (12,6)
#filepath = './DGBQA_HGR_Res3D-ViViT_SOLI.pdf'
#comparitive_bars(eer_score,dgbqa_score_norm,num_gestures,gesture_list,filepath)

###### Avg. Rank Deviation
#avg_rank_dev = avg_rank_deviation(eer_score,dgbqa_score_norm,num_gestures)
#print('Avg. Rank Deviation: '+str(avg_rank_dev))

###### CGID Score Computations
#plt.rcParams["figure.figsize"] = [72,72]
#filepath = './DGBQA_HGR_Res3D-ViViT_HandLogin.pdf' 
#cgid_score_decorr, cgid_score = CGID_Score_Calculator(Test_Embeddings,y_dev,filepath)
#print('CGID Score: '+str(round(cgid_score,3)))              
#print('CGID Score Decorrelated: '+str(round(cgid_score_decorr,3)))

Inter-Distance - d_c_star: 0.5013526347776254
Intra-Distance - d_cs_avg: 0.6298872213810682
Inter-Distance - d_c_star: 0.6765149756024281
Intra-Distance - d_cs_avg: 0.6615247540175915
Inter-Distance - d_c_star: 0.7872827542324861
Intra-Distance - d_cs_avg: 0.6384690115228295
Inter-Distance - d_c_star: 0.9045808727542559
Intra-Distance - d_cs_avg: 0.6362616270780563
[-0.3769924610633828, 0.037261142540255365, 0.34947880259402064, 0.6043872990155489]


## CGID-Score

In [8]:
####### Mask Generation
###### Positive Mask
def get_positive_mask(labels):
        """
        Return a 2D mask where mask[a, p] is True iff a and p are distinct and have same label.
        Args:
            labels: tf.int32 `Tensor` with shape [batch_size]
        Returns:
            mask: tf.bool `Tensor` with shape [batch_size, batch_size]
        """
        # Check that i and j are distinct
        indices_equal = tf.cast(tf.eye(labels.shape[0]), tf.bool)
        indices_not_equal = tf.logical_not(indices_equal)

        # Check if labels[i] == labels[j]
        # Uses broadcasting where the 1st argument has shape (1, batch_size) and the 2nd (batch_size, 1)
        labels_equal = tf.equal(tf.expand_dims(labels, 0), tf.expand_dims(labels, 1))

        # Combine the two masks``
        mask = tf.logical_and(indices_not_equal, labels_equal)

        return mask
    
###### Negative Mask - Different Mask
def get_negative_mask(labels):
    """Return a 2D mask where mask[a, n] is True iff a and n have distinct labels.
    Args:
        labels: tf.int32 `Tensor` with shape [batch_size]
    Returns:
        mask: tf.bool `Tensor` with shape [batch_size, batch_size]
    """
    # Check if labels[i] != labels[k]
    # Uses broadcasting where the 1st argument has shape (1, batch_size) and the 2nd (batch_size, 1)
    labels_equal = tf.equal(tf.expand_dims(labels, 0), tf.expand_dims(labels, 1))

    mask = tf.logical_not(labels_equal)

    return mask

###### Heatmap Plotting Function
def plot_heatmap(cm,filepath,title='CGID Gramian Matrix',cmap=plt.cm.Blues):
    
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(filepath)
    plt.close()

####### CGID Score
def CGID_Score_Calculator(f_theta,y_hgr,filepath):
      
    """
    Function to Compute CGID Score and Plot correponding Gramian Matrix Heatmap
          
    INPUTS:-
    1) f_theta: Embeddings of the shape (N,d); N-Total Examples, d: Embedding Dimensions
    2) y_hgr: HGR labels of the shape (N,)
    3) filepath: Path to save the plotted 
    
    OUTPUTS:-
    1) cgid_score_decorr: Average of gram matrix computed over the all the examples
    2) cgid_score: Average of masked gram matrix
    """

    ##### L2-Normalization
    f_theta = tf.math.l2_normalize(f_theta,axis=1)

    ##### Different Gesture Mask Computation
    delta_bar = get_negative_mask(y_hgr)
    delta_bar = np.reshape(delta_bar.numpy(),(delta_bar.shape[0],delta_bar.shape[1]))

    ##### Gramian Matrix Formation
    G_bar = tf.cast(tf.linalg.matmul(f_theta,f_theta,transpose_b=True),dtype=tf.float32)
    cgid_score_matrix = np.multiply(delta_bar,G_bar.numpy())
    cgid_score_matrix_mask = (cgid_score_matrix >= 0)
    plot_heatmap(cgid_score_matrix*cgid_score_matrix_mask,filepath)

    ##### CGID-Score
    cgid_score = (np.sum(cgid_score_matrix*cgid_score_matrix_mask))/(np.sum(cgid_score_matrix_mask))
    cgid_score_decorr = np.mean(cgid_score_matrix)
    
    return cgid_score_decorr, cgid_score

# Scoring Frameworks

In [9]:
##################################################################################################################
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
### Scoring Frameworks
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
##################################################################################################################

###### Delta-Distance Score
def delta_dist_compute(embeddings,g_id,num_subjects,y_dev,y_dev_id):

    """
    Function to Compute Delta Distance for a Gesture

    INPUTS:-
    1) embeddings: Feature Embeddings
    2) g_id: Index Value of the Gesture Class
    3) num_subjects: Number of Subjects
    4) y_dev: Ground-Truth Gesture Labels
    5) y_dev_id: Ground-Truth Subject Labels

    OUTPUTS:-
    1) delta_dist: |d_cs - d_c|/d_c, Delta Distance Computed for a particular gesture

    """
    ###### Defining Essentials
    d_cs = [] # List to Maximal Intra-Subject Distance

    ###### Iterating over Subjects
    for s_id in range(num_subjects): 

        ##### Defining Essentials
        embedding_store_s = [] # List to Store Embeddings from Gesture - 'g_id' and Subject - 's_id'
        dist_store_s = [] # List to Store Distance within subject 's_id' 

        ##### Curating Required Gesture List from g_id and s_id
        for idx in range(y_dev.shape[0]): # Iterating over Embeddings

            if(y_dev[idx] == g_id and y_dev_id[idx] == s_id):
                embedding_store_s.append(embeddings[idx]) # Storing the Required Embeddings
        
        ##### Computing Intra-Gesture and Intra-Subject Distances
        for emb_query_idx, emb_query in enumerate(embedding_store_s):

            if(emb_query_idx != (len(embedding_store_s)-1)): # Checking if the Current Query is the Last Query

                for emb_key_idx in range(emb_query_idx+1,len(embedding_store_s),1): # Iterating over the Embeddings

                    emb_key_curr = embedding_store_s[emb_key_idx] # Extracting Current Embedding Key
                    dist_curr = distance.euclidean(emb_query,emb_key_curr) # Current Distance 
                    dist_store_s.append(dist_curr) # Appending the Computed Distance to dist_curr

        ##### Computing Maximal Distance for the Current Gesture and Subject
        d_cs_curr = np.max(dist_store_s)
        d_cs.append(d_cs_curr) # Storing Values

    ###### Computing Delta Distance
    d_c = np.max(d_cs) # Maximal Distance Amongst all the Subjects
    print('Average Intra-Subject Separation: '+str(np.average(d_cs)))
    print('Minimum Intra-Subject Separation: '+str(np.min(d_cs)))
    print('Maximum Intra-Subject Separation (d_cs): '+str(d_c))
    delta_dist = np.average(np.abs(d_cs - d_c)/d_c) # Averaging Computed Delta Distance Per Subject
    
    return delta_dist

###### Iterating over Gestures
delta_distance = []
for g_id, gesture_curr in enumerate(gesture_list):
    print('==============================================')
    delta_dist_g = delta_dist_compute(Test_Embeddings,g_id,num_subjects,y_dev,y_dev_id)
    delta_distance.append(delta_dist_g)
    print('Delta Distance for '+str(gesture_curr)+' = '+str(delta_dist_g))

delta_distance = np.array(delta_distance) # Array Formation
delta_distance = (delta_distance - np.mean(delta_distance))/np.std(delta_distance) # Mean Normalization
delta_distance = delta_distance/np.linalg.norm(delta_distance) # L2-Normalization

####### Generative Capacity
###### Cosine Bounds Function
def get_cosine_bounds(X, quantile=0.05):
    cosine_dist = np.dot(X, X.transpose())
    min_val = np.min(cosine_dist, axis=1)

    mask = np.ones(cosine_dist.shape, dtype=bool)
    np.fill_diagonal(mask, 0)
    cosine_dist = cosine_dist * mask

    max_val = np.max(cosine_dist, axis=1)
    range_val = max_val - min_val

    value = np.quantile(min_val, quantile)
    total_angle = np.arccos(value) * 180 / np.pi

    return total_angle, min_val, max_val

###### Capacity Estimation Function
def ratio_hyperspherical_caps(inter_class_angle, intra_class_angle, cos_delta, sin_delta, d):
    
    # compute cos(\theta) where \theta is the solid
    # angle corresponding to the inter-class hyper-spherical cap
    cos_theta = np.cos(inter_class_angle)
    sin_theta = np.sqrt(1 - cos_theta**2)
    
    cos_omega = cos_theta * cos_delta - sin_theta * sin_delta
     
    x = 1 - cos_omega * cos_omega
    a = (d - 1)/2
    b = 0.5 

    numerator = sp.betainc(a, b, x)
    index = np.where(cos_omega < 0)
    #numerator[index] = 0.5 + numerator[index]
    
    # compute cos(\phi) where \phi is the solid
    # angle corresponding to the intra-class hyper-spherical cap
    cos_phi = np.cos(intra_class_angle)
    sin_phi = np.sqrt(1 - cos_phi**2)
    cos_omega = cos_phi * cos_delta - sin_phi * sin_delta

    x = 1 - cos_omega * cos_omega
    a = (d - 1)/2
    b = 0.5

    denominator = sp.betainc(a, b, x)
    index = np.where(cos_omega < 0)
    #denominator[index] = 0.5 + denominator[index]

    capacity = numerator / denominator
    #capacity[capacity < 1] = 1
    
    return capacity

####### Biometric Capacity
###### Defining Essentials
total_gestures = 4
total_ids = 16
g_angle = [] # List to store Intra-Gesture Angular Capacity
g_id_angle = [] # List to store Intra-Gesture Id Angular Capacity
Capacity_Value = [] # List to store Generative Biometric Capacity of each of the gesture
d_size = 32 # Embedding Size
delta = 0 # Setting Delta(FAR Parameter) to zero

###### Iteration Loop
for gesture_val in range(total_gestures): # Iterating over the Gestures

    ###### Gesture-level
    X_store = [] # List to store all the examples of that gesture
    #idx_store = [] # List to store all the indexes of the gestures being stored
    id_store = [] # List to store the identity-labels corresponding to the gesture
    g_id_angle_store = [] # List to store Angular Spreads of the 'N' identities involved in the dataset

    ##### Gesture-Store Curation
    for g_idx, X_ges in enumerate(Test_Embeddings): # Iterating over the features

        if(y_dev[g_idx] == gesture_val): # Checking for the Gesture Labels

            X_store.append(X_ges) # Storing the Feature
            id_store.append(y_dev_id[g_idx]) # Storing ID-label of the feature

    ##### Estimation of Gesture-Level Angular Spread
    g_angle_curr,_,_ = get_cosine_bounds(np.array(X_store))
    g_angle_curr = (g_angle_curr/2)*(np.pi/180)
    g_angle.append(g_angle_curr)

    ##### Estimation of Intra-Gesture Id-Level Angular Spread
    for id_idx in range(total_ids): # Searching for Particular Identities
        X_id_store = [] # List to store Intra-Gesture features of a particular identity

        for item_idx, item in enumerate(X_store): # Iterating over the Current Gesture-Store
            
            if(id_store[item_idx] == id_idx): # Identity Match-found
                X_id_store.append(item) # Storing the Feature

        #### Estimation of Intra-Gesture Intra-Id Angular Spread
        g_id_angle_curr,_,_ = get_cosine_bounds(np.array(X_id_store))
        g_id_angle_curr = (g_id_angle_curr/2)*(np.pi/180)    
        g_id_angle_store.append(g_id_angle_curr)

    g_id_angle_curr_overall = np.average(g_id_angle_store) # Avg. Angular Spread of all the Identities within the gesture under consideration
    g_id_angle.append(g_id_angle_curr_overall) # Storing in Global List

    ##### Estimation of Gesture's Biometric Capacity
    capacity_curr = ratio_hyperspherical_caps(g_angle_curr,g_id_angle_curr_overall,1,0,d_size) # Biometric Capacity of the Current Gesture
    Capacity_Value.append(capacity_curr) # Storing Values

#print(Capacity_Value)
Capacity_Value = np.array(Capacity_Value)
Capacity_Value = (Capacity_Value - np.mean(Capacity_Value))/np.std(Capacity_Value)
Capacity_Value = Capacity_Value/np.linalg.norm(Capacity_Value)

####### MasterFace Capacity
def Compute_MasterFace_Capacity(d_c_star,d_size): 

    """
    Function to Compute MasterFace based Gesture Capacity

    INPUTS:-
    1) d_c_star: Avg. Distance between different Identity Centroids within a Gesture
    2) d_size: Size of the Embeddings

    OUTPUTS:-
    1) MasterFace_Capacity: Estimated Biometric Capacity within Hand-Gestures  
    """

    MasterFace_Capacity = np.exp((d_size*(0.993 - 0.436*(1-d_c_star))+3.701-3.706*(1 - d_c_star)))
    return MasterFace_Capacity

MasterFace_Capacity = Compute_MasterFace_Capacity(np.array(d_c_star),32)
MasterFace_Capacity = (MasterFace_Capacity - np.mean(MasterFace_Capacity))/np.std(MasterFace_Capacity)
MasterFace_Capacity = MasterFace_Capacity/np.linalg.norm(MasterFace_Capacity)

Average Intra-Subject Separation: 0.6298872213810682
Minimum Intra-Subject Separation: 0.36934810876846313
Maximum Intra-Subject Separation (d_cs): 0.8029829263687134
Delta Distance for Compass = 0.21556585987503693
Average Intra-Subject Separation: 0.6615247540175915
Minimum Intra-Subject Separation: 0.2828809916973114
Maximum Intra-Subject Separation (d_cs): 1.0082943439483643
Delta Distance for Piano = 0.3439170238453021
Average Intra-Subject Separation: 0.6384690115228295
Minimum Intra-Subject Separation: 0.24417833983898163
Maximum Intra-Subject Separation (d_cs): 1.2539219856262207
Delta Distance for Push = 0.49082238062524125
Average Intra-Subject Separation: 0.6362616270780563
Minimum Intra-Subject Separation: 0.3050422668457031
Maximum Intra-Subject Separation (d_cs): 1.0302234888076782
Delta Distance for Flipping Fist = 0.38240427053897874


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22400\3466457391.py:109: DeprecationWarning: Calling nonzero on 0d arrays is deprecated, as it behaves surprisingly. Use `atleast_1d(cond).nonzero()` if the old behavior was intended. If the context of this warning is of the form `arr[nonzero(cond)]`, just use `arr[cond]`.
  index = np.where(cos_omega < 0)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_22400\3466457391.py:123: DeprecationWarning: Calling nonzero on 0d arrays is deprecated, as it behaves surprisingly. Use `atleast_1d(cond).nonzero()` if the old behavior was intended. If the context of this warning is of the form `arr[nonzero(cond)]`, just use `arr[cond]`.
  index = np.where(cos_omega < 0)


In [ ]:
####### CGID Score Computations
plt.rcParams["figure.figsize"] = [72,72]
Test_Embeddings = np.load('/kaggle/input/dgbqa-vanilla-res3d-vivit-iar-soli/DGBQA_Vanilla_Res3D-ViViT_1-1_IAR_SOLI.npz',allow_pickle=True)['arr_0']
y_dev = np.load('/kaggle/input/rhga-deltadistance-soli-dataset/y_dev_DeltaDistance_SOLI.npz',allow_pickle=True)['arr_0']
filepath = './DGBQA_Vanilla+CGID_Res3D-ViViT_1-0_CGID-Matrix.png'
cgid_score_decorr, cgid_score = CGID_Score_Calculator(Test_Embeddings,y_dev,filepath)
print('CGID Score Decorrelated: '+str(round(cgid_score_decorr,3)))
print('CGID Score: '+str(round(cgid_score,3)))

# Evaluation Measures

In [10]:
##################################################################################################################
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
### Performance Measures
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
##################################################################################################################

###### CGID and Decorrelated-CGID Score

##### Mask Generation
#### Positive Mask
def get_positive_mask(labels):
        """
        Return a 2D mask where mask[a, p] is True iff a and p are distinct and have same label.
        Args:
            labels: tf.int32 `Tensor` with shape [batch_size]
        Returns:
            mask: tf.bool `Tensor` with shape [batch_size, batch_size]
        """
        # Check that i and j are distinct
        indices_equal = tf.cast(tf.eye(labels.shape[0]), tf.bool)
        indices_not_equal = tf.logical_not(indices_equal)

        # Check if labels[i] == labels[j]
        # Uses broadcasting where the 1st argument has shape (1, batch_size) and the 2nd (batch_size, 1)
        labels_equal = tf.equal(tf.expand_dims(labels, 0), tf.expand_dims(labels, 1))

        # Combine the two masks``
        mask = tf.logical_and(indices_not_equal, labels_equal)

        return mask
    
###### Negative Mask - Different Mask
def get_negative_mask(labels):
    """Return a 2D mask where mask[a, n] is True iff a and n have distinct labels.
    Args:
        labels: tf.int32 `Tensor` with shape [batch_size]
    Returns:
        mask: tf.bool `Tensor` with shape [batch_size, batch_size]
    """
    # Check if labels[i] != labels[k]
    # Uses broadcasting where the 1st argument has shape (1, batch_size) and the 2nd (batch_size, 1)
    labels_equal = tf.equal(tf.expand_dims(labels, 0), tf.expand_dims(labels, 1))

    mask = tf.logical_not(labels_equal)

    return mask

###### Heatmap Plotting Function
def plot_heatmap(cm,filepath,title='CGID Gramian Matrix',cmap=plt.cm.Blues):
    
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(filepath)
    plt.close()

####### CGID Score
def CGID_Score_Calculator(f_theta,y_hgr):
      
    """
    Function to Compute CGID Score and Plot correponding Gramian Matrix Heatmap
          
    INPUTS:-
    1) f_theta: Embeddings of the shape (N,d); N-Total Examples, d: Embedding Dimensions
    2) y_hgr: HGR labels of the shape (N,)
    3) filepath: Path to save the plotted 
    
    OUTPUTS:-
    1) cgid_score_decorr: Average of gram matrix computed over the all the examples
    2) cgid_score: Average of masked gram matrix
    """

    ##### L2-Normalization
    f_theta = tf.math.l2_normalize(f_theta,axis=1)

    ##### Different Gesture Mask Computation
    delta_bar = get_negative_mask(y_hgr)
    delta_bar = np.reshape(delta_bar.numpy(),(delta_bar.shape[0],delta_bar.shape[1]))

    ##### Gramian Matrix Formation
    G_bar = tf.cast(tf.linalg.matmul(f_theta,f_theta,transpose_b=True),dtype=tf.float32)
    cgid_score_matrix = np.multiply(delta_bar,G_bar.numpy())
    cgid_score_matrix_mask = (cgid_score_matrix >= 0)
    #plot_heatmap(cgid_score_matrix*cgid_score_matrix_mask,filepath)

    ##### CGID-Score
    cgid_score = (np.sum(cgid_score_matrix*cgid_score_matrix_mask))/(np.sum(cgid_score_matrix_mask))
    cgid_score_decorr = np.mean(cgid_score_matrix)
    
    return cgid_score_decorr, cgid_score

####### Rank-Deviation
###### Rank Derivation
def rank_compute(rank_vector,val_to_rank,reverse_flag):

    """
    Function to derive rank of a particular value

    INPUTS:-
    1) rank_vector: Vector in which ranking is to be performed
    2) val_to_rank: Value whoose rank is to be derived
    3) reverse_flag: Flag to signify if the order of sort is to be reversed

    OUTPUTS:-
    1) rank_val: Ranked Value - The best is rank '0'
    """ 

    if(reverse_flag == True):
        rank_vector_sort = -(np.sort(-rank_vector))
    else:
        rank_vector_sort = np.sort(rank_vector) # Sorting the Vector    
    for item_idx,item in enumerate(rank_vector_sort): # Iterating over the vector
        if(item == val_to_rank): # Match-Found
            rank_val = item_idx # Rank Assignment 
            break

    return rank_val

###### Avg. Rank Deviation
def avg_rank_deviation(eer_vec,dgbqa_vec,G_Total):
    
    """
    Computation of Acceptance Score
    
    INPUTS:-
    1) eer_vec: Vector of EER values
    2) dgbqa_vec: Vector of DGBQA values
    3) G_Total: Total gestures in the vector
    
    OUTPUTS:-
    1) avg_rank_deviation: Total Deviation in Rank/Total Number of Gestures
    """
    
    rank_deviation = 0 # Total Rank Deviation
    
    for g_idx in range(G_Total):
        
        #### Values of the Current Gesture
        eer_val_curr = eer_vec[g_idx] # EER Value
        dgbqa_val_curr = dgbqa_vec[g_idx] # DGBQA Value
        
        #### Rank of the Current Gesture
        eer_rank_curr = rank_compute(eer_vec,eer_val_curr,True) # EER Rank
        dgbqa_rank_curr = rank_compute(dgbqa_vec,dgbqa_val_curr,False) # DGBQA Rank
        
        #### Rank Difference
        rank_deviation =  rank_deviation + np.abs(eer_rank_curr - dgbqa_rank_curr)
        
    avg_rank_deviation = rank_deviation/G_Total # Computing Avg. over all the gestures
    return avg_rank_deviation

####### Helper Functions
###### Rank Computation
def rank_compute_acc(rank_vector,val_to_rank):

    """
    Function to derive rank of a particular value

    INPUTS:-
    1) rank_vector: Sorted Vector in which ranking is to be performed
    2) val_to_rank: Value whoose rank is to be derived

    OUTPUTS:-
    1) rank_val: Ranked Value - The best is rank '1', but we use index of '0' all along but just the Relevance Function
    """
    #rank_vector_sort = -np.sort(-rank_vector) # Sorting the Vector: The biggest value is the better-most value

    for item_idx,item in enumerate(rank_vector): # Iterating over the vector
        if(item == val_to_rank): # Match-Found
            rank_val = item_idx # Rank Assignment 
            break

    return rank_val

####### Acceptance Score
def acceptance_score(dgbqa,e_prime,G,normalizer=False):
 
    """
    Function to compute Acceptance Score: Sum over all Gestures(Relevance/Rank Deviation)

    INPUTS:-
    1) dgbqa: Array of unranked but normalized DGBQA score values
    2) e_prime: Array of unranked but normalized (1 - EER) values
    3) G: Total number of gestures considered for analysis
    3) normalizer: If True, Ar for e_prime will be computed. Default value = False 

    OUTPUTS:-
    1) Ar: Acceptance Score: Sum over all Gestures(Relevance/Rank Deviation)
    """

    ##### Defining Essentials
    e_prime_sort = -(np.sort(-e_prime)) # Sorted e_prime
    dgbqa_sort = -(np.sort(-dgbqa)) # Sorted DGBQA-Score Values
    dgbqa_re = [] # DGBQA-Scores Ordered as per the e_prime_sort  
    arrangement_idx = [] # List to store arrangement orders of e_prime, with the values starting from 0, and ending at G-1
    Ar = 0 # Initializing Acceptance Value as zero
    lambda_scale = 2 # Scaling factor for relevance
    gamma = 2 # Scaling Factor for the first term in relevance formulation
    kappa = 1  # Scaling Factor for the Rank-Deviation Penalty

    if(normalizer == False): # Checking if the Ar is being estimated for the normalizer or not

        ##### Aranging DGBQA Scores as per e_prime's order
        for idx, val in enumerate(e_prime_sort):

            for idx_search, val_search in enumerate(e_prime):

                if(val == val_search):
                    arrangement_idx.append(idx_search) # Finding Index on e_prime that matches with e_prime_sort
                    dgbqa_re.append(dgbqa[idx_search]) # Appending terms in dgbqa_re as per the order in e_prime_sort: The best rank is the first term
                    break

            #print(arrangement_idx)

        ##### Ar Estimation
        for r_e_j, dgbqa_r_e_j in enumerate(dgbqa_re): # Iterating over all the gestures in the set

            #### Relevance Gain
            R_j = gamma*((G-(r_e_j+1)+1)/G)*dgbqa_r_e_j + ((r_e_j+1)/G)*(1 - dgbqa_r_e_j)
            R_j = 2**(lambda_scale*R_j)

            #### Rank Deviation Penalty
            ### Rank Computation
            rank_dgbqa = rank_compute_acc(np.array(dgbqa_sort),dgbqa_r_e_j) # Rank Derived as per DGBQA-Score Estimates
            rank_e_prime = rank_compute_acc(np.array(dgbqa_re),dgbqa_r_e_j) # Rank Derived as per e-prime-sort based sorting of DGBQA-Scores

            ### Rank Deviation
            rank_dev_j = np.exp(kappa*np.abs(rank_dgbqa - rank_e_prime)) 

            #### Ar Estimate for the Current Gesture
            Ar_j = R_j/rank_dev_j
            Ar = Ar + Ar_j # Adding this to the Metric
            #print(r_e_j,R_j,np.abs(rank_dgbqa - rank_e_prime),rank_dev_j,Ar_j)

        return Ar
     
    else:

        ##### Ar Estimation
        for r_e_j, eprime_r_e_j in enumerate(e_prime_sort): # Iterating over all the gestures in the set
            
            #### Relevance Gain
            R_j = gamma*((G - (r_e_j+1)+1)/G)*eprime_r_e_j + ((r_e_j+1)/G)*(1 - eprime_r_e_j)
            #print(r_e_j,R_j)
            R_j = 2**(lambda_scale*R_j)
            #print(r_e_j,R_j)

            #### Ar Estimate for the Current Gesture
            Ar_j = R_j
            Ar = Ar + Ar_j # Adding this to the Metric

        return Ar

####### Pattern-Matching Distance 
def pattern_match_dist(dgbqa,e_prime,G):

    """
    Function to compute Pattern-Match Distance

    INPUTS:-
    1) dgbqa: Array of unranked but normalized DGBQA score values
    2) e_prime: Array of unranked but normalized (1 - EER) values
    3) G: Total number of gestures considered for analysis

    OUTPUTS:-
    1) pm_dist: Pattern-Match Distance
    """
    ##### Defining Essentials
    e_prime_sort = -np.sort(-np.array(e_prime)) # Sorting e_prime
    dgbqa_re = [] # DGBQA-Scores Ordered as per the e_prime_sort
    arrangement_idx = [] # List to store arrangement orders of e_prime, with the values starting from 0, and ending at G-1
    pm_dist = 0 # Initializing pm_dist as 0
    dgbqa_re_f = [] # Low-to-High Ranked List of DGBQA-Score sorted as per e_prime  
    e_prime_sort_f = [] # Low-to-High Ranked List of e_prime values

    ##### e_prime in reverse order
    for idx in range(len(e_prime_sort)):
        e_prime_sort_f.append(e_prime_sort[len(e_prime_sort)-idx-1])

    e_prime_sort_f = np.array(e_prime_sort_f) # Array Formation

    ##### Aranging DGBQA Scores as per e_prime's order
    for idx, val in enumerate(e_prime_sort):

        for idx_search, val_search in enumerate(e_prime):

            if(val == val_search):
                arrangement_idx.append(idx_search) # Finding Index on e_prime that matches with e_prime_sort
                dgbqa_re.append(dgbqa[idx_search]) # Appending terms in dgbqa_re as per the order in e_prime_sort: The best rank is the first term
                break

    dgbqa_re_b = np.array(dgbqa_re) # Array Formation
    
    ##### Aranging DGBQA Scores as per e_prime's order in Low to High
    for idx in range(len(dgbqa_re)):
        dgbqa_re_f.append(dgbqa_re[len(dgbqa_re)-idx-1])

    dgbqa_re_f = np.array(dgbqa_re_f) # Array Formation 

    ##### Pattern-Matching Distance Estimation

    #### Slope Computation
    tan_theta_f = dgbqa_re_f[1:] - dgbqa_re_f[:-1] # Forward Movement Slopes
    tan_theta_b = tan_theta_f # Backward Movement Slope is Similar as the Forward Movement Slope, just estimation method differs

    #### Forward Value Computation
    ### Estimation
    e_bar_f = e_prime_sort_f[:-1] + tan_theta_f # e2_bar_f to eG_bar_f
    eG_bar_f = e_bar_f[-1] # Extracting Last Value
    e_bar_f = e_bar_f[:-1] # Storing Middle 'G-2' Values

    ### Error Computation
    error_f = np.sum(np.abs(e_bar_f - e_prime_sort_f[1:-1])) + 2*(np.abs(eG_bar_f - e_prime_sort_f[-1]))

    #### Backward Value Computation
    ### Estimation
    e_bar_b = e_prime_sort_f[1:] - tan_theta_f # e1_bar to e(G-1)_bar_f
    e1_bar_b = e_bar_b[0] # Extracting First Value
    e_bar_b = e_bar_b[1:] # Storing Middle 'G-2' Values

    ### Error Compuation
    error_b = np.sum(np.abs(e_bar_b - e_prime_sort_f[1:-1])) + 2*(np.abs(e1_bar_b - e_prime_sort_f[0])) 
    
    #### Pattern-Matching Distance
    pm_dist = 0.5*(error_f + error_b)

    return pm_dist

####### Acceptance Score
def acceptance_score_comp(dgbqa,e_prime,G):
 
    """
    Function to compute Acceptance Score: Sum over all Gestures(Relevance/Rank Deviation)

    INPUTS:-
    1) dgbqa: Array of unranked but normalized DGBQA score values
    2) e_prime: Array of unranked but normalized (1 - EER) values
    3) G: Total number of gestures considered for analysis

    OUTPUTS:-
    1) Ar: Acceptance Score: Sum over all Gestures(Relevance/Rank Deviation)
    """

    ##### Defining Essentials
    e_prime_sort = -(np.sort(-e_prime)) # Sorted e_prime
    dgbqa_sort = -(np.sort(-dgbqa)) # Sorted DGBQA-Score Values
    dgbqa_re = [] # DGBQA-Scores Ordered as per the e_prime_sort  
    arrangement_idx = [] # List to store arrangement orders of e_prime, with the values starting from 0, and ending at G-1
    Ar = 0 # Initializing Acceptance Value as zero
    lambda_scale = 2 # Scaling factor for relevance
    gamma = 2 # Scaling Factor for the first term in relevance formulation
    kappa = 1  # Scaling Factor for the Rank-Deviation Penalty

    ##### Aranging DGBQA Scores as per e_prime's order
    for idx, val in enumerate(e_prime_sort):

        for idx_search, val_search in enumerate(e_prime):

            if(val == val_search):
                arrangement_idx.append(idx_search) # Finding Index on e_prime that matches with e_prime_sort
                dgbqa_re.append(dgbqa[idx_search]) # Appending terms in dgbqa_re as per the order in e_prime_sort: The best rank is the first term
                break

        #print(arrangement_idx)

    ##### Ar Estimation
    for r_e_j, dgbqa_r_e_j in enumerate(dgbqa_re): # Iterating over all the gestures in the set

        #### Relevance Gain
        #R_j = gamma*((G-(r_e_j+1)+1)/G)*dgbqa_r_e_j + ((r_e_j+1)/G)*(1 - dgbqa_r_e_j)
        #R_j = 2**(lambda_scale*R_j)

        #### Rank Deviation Penalty
        ### Rank Computation
        rank_dgbqa = rank_compute_acc(np.array(dgbqa_sort),dgbqa_r_e_j) # Rank Derived as per DGBQA-Score Estimates
        rank_e_prime = rank_compute_acc(np.array(dgbqa_re),dgbqa_r_e_j) # Rank Derived as per e-prime-sort based sorting of DGBQA-Scores

        ### Rank Deviation
        rank_dev_j = np.exp(kappa*np.abs(rank_dgbqa - rank_e_prime)) 

        #### Ar Estimate for the Current Gesture
        Ar_j = 1/rank_dev_j
        Ar = Ar + Ar_j # Adding this to the Metric
        #print(r_e_j,R_j,np.abs(rank_dgbqa - rank_e_prime),rank_dev_j,Ar_j)1

    return Ar

# EER Processing

In [11]:
##################################################################################################################
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
### EER-Processing
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
##################################################################################################################

####### EER-Processing
e = [0.44,1.29,4.89,1.05]
e = np.array(e)
e_prime = 100 - np.array(e)
e_prime = (e_prime - np.mean(e_prime))/np.std(e_prime)
e_prime = e_prime/np.linalg.norm(e_prime)

# Feature-Space Scores

In [10]:
##################################################################################################################
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
### Feature-Space Scores
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
##################################################################################################################

####### CGID and Decorr-CGID Score
C_I, C_D = CGID_Score_Calculator(Test_Embeddings,y_dev)
print('===============================================')
print('CGID Score: '+str(round(C_I,3)))              
print('CGID Score Decorrelated: '+str(round(C_D,3)))

CGID Score: 0.371
CGID Score Decorrelated: 0.371


# Comparison Scores

In [13]:
##################################################################################################################
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
### DGBQA-Scores
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
##################################################################################################################
####### DGBQA Score
###### Defining Essentials
gesture_list = ['Compass','Piano','Push','Flipping Fist']
num_subjects = 16
num_gestures = 4
dgbqa_score = []
d_c_star = []
d_cs = []
dgbqa_score_wo = []
#eer_score = np.array([0.44,1.29,4.89,1.05])
#eer_score = eer_score/np.linalg.norm(eer_score)
Test_Embeddings = np.load('./Embeddings/DGBQA_CGID_Res3D-MF_1-pt5_HandLogin.npz',allow_pickle=True)['arr_0']

###### Iterating over Gestures
for g_id, gesture_curr in enumerate(gesture_list):
    print('==============================================')
    dgbqa_score_curr, d_c_star_curr, d_cs_curr, dgbqa_score_wo_curr = gbqa_delta_dist_compute(Test_Embeddings,g_id,num_subjects,y_dev,y_dev_id)
    dgbqa_score.append(dgbqa_score_curr)
    d_c_star.append(d_c_star_curr)
    d_cs.append(d_cs_curr)
    dgbqa_score_wo.append(dgbqa_score_wo_curr)
    #print('GBQA Delta Distance for '+str(gesture_curr)+' = '+str(dgbqa_score_curr))  

print(dgbqa_score)

dgbqa_score = np.array(dgbqa_score) # Array Formation
dgbqa_score = (dgbqa_score - np.mean(dgbqa_score))/np.std(dgbqa_score) # Mean Normalization
dgbqa_score = dgbqa_score/np.linalg.norm(dgbqa_score) # L2-Normalization

d_c_star = np.array(d_c_star) # Array Formation
d_c_star = (d_c_star - np.mean(d_c_star))/np.std(d_c_star) # Mean Normalization
d_c_star = d_c_star/np.linalg.norm(d_c_star) # L2-Normalization

d_cs = np.array(d_cs) # Array Formation
d_cs = (d_cs - np.mean(d_cs))/np.std(d_cs) # Mean Normalization
d_cs = d_cs/np.linalg.norm(d_cs) # L2-Normalization

dgbqa_score_wo = np.array(dgbqa_score_wo) # Array Formation
dgbqa_score_wo = (dgbqa_score_wo - np.mean(dgbqa_score_wo))/np.std(dgbqa_score_wo) # Mean Normalization
dgbqa_score_wo = dgbqa_score_wo/np.linalg.norm(dgbqa_score_wo) # L2-Normalization

###### Acceptance Score
#dgbqa_score_norm = dgbqa_score/np.linalg.norm(dgbqa_score)
#acceptance_score = -(np.sum(np.multiply(eer_score,dgbqa_score_norm)))
#ranked_acceptance_score = compute_acceptance_score(eer_score,dgbqa_score_norm,4)
#print('==============================================')
#print('Acceptance-Score: '+str(acceptance_score))
#print('Acceptance-Score-Norm: '+str(acceptance_score/num_gestures)) 
#print('Ranked Acceptance-Score: '+str(ranked_acceptance_score))

###### Comparitive Bar Graphs
#plt.rcParams["figure.figsize"] = (12,6)
#filepath = './DGBQA_HGR_Res3D-ViViT_SOLI.pdf'
#comparitive_bars(eer_score,dgbqa_score_norm,num_gestures,gesture_list,filepath)

###### Avg. Rank Deviation
#avg_rank_dev = avg_rank_deviation(eer_score,dgbqa_score_norm,num_gestures)
#print('Avg. Rank Deviation: '+str(avg_rank_dev))

###### CGID Score Computations
#plt.rcParams["figure.figsize"] = [72,72]
#filepath = './DGBQA_HGR_Res3D-ViViT_HandLogin.pdf' 
#cgid_score_decorr, cgid_score = CGID_Score_Calculator(Test_Embeddings,y_dev,filepath)
#print('CGID Score: '+str(round(cgid_score,3)))              
#print('CGID Score Decorrelated: '+str(round(cgid_score_decorr,3)))

##################################################################################################################
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
### Scoring Frameworks
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
##################################################################################################################

###### Delta-Distance Score
def delta_dist_compute(embeddings,g_id,num_subjects,y_dev,y_dev_id):

    """
    Function to Compute Delta Distance for a Gesture

    INPUTS:-
    1) embeddings: Feature Embeddings
    2) g_id: Index Value of the Gesture Class
    3) num_subjects: Number of Subjects
    4) y_dev: Ground-Truth Gesture Labels
    5) y_dev_id: Ground-Truth Subject Labels

    OUTPUTS:-
    1) delta_dist: |d_cs - d_c|/d_c, Delta Distance Computed for a particular gesture

    """
    ###### Defining Essentials
    d_cs = [] # List to Maximal Intra-Subject Distance

    ###### Iterating over Subjects
    for s_id in range(num_subjects): 

        ##### Defining Essentials
        embedding_store_s = [] # List to Store Embeddings from Gesture - 'g_id' and Subject - 's_id'
        dist_store_s = [] # List to Store Distance within subject 's_id' 

        ##### Curating Required Gesture List from g_id and s_id
        for idx in range(y_dev.shape[0]): # Iterating over Embeddings

            if(y_dev[idx] == g_id and y_dev_id[idx] == s_id):
                embedding_store_s.append(embeddings[idx]) # Storing the Required Embeddings
        
        ##### Computing Intra-Gesture and Intra-Subject Distances
        for emb_query_idx, emb_query in enumerate(embedding_store_s):

            if(emb_query_idx != (len(embedding_store_s)-1)): # Checking if the Current Query is the Last Query

                for emb_key_idx in range(emb_query_idx+1,len(embedding_store_s),1): # Iterating over the Embeddings

                    emb_key_curr = embedding_store_s[emb_key_idx] # Extracting Current Embedding Key
                    dist_curr = distance.euclidean(emb_query,emb_key_curr) # Current Distance 
                    dist_store_s.append(dist_curr) # Appending the Computed Distance to dist_curr

        ##### Computing Maximal Distance for the Current Gesture and Subject
        d_cs_curr = np.max(dist_store_s)
        d_cs.append(d_cs_curr) # Storing Values

    ###### Computing Delta Distance
    d_c = np.max(d_cs) # Maximal Distance Amongst all the Subjects
    print('Average Intra-Subject Separation: '+str(np.average(d_cs)))
    print('Minimum Intra-Subject Separation: '+str(np.min(d_cs)))
    print('Maximum Intra-Subject Separation (d_cs): '+str(d_c))
    delta_dist = np.average(np.abs(d_cs - d_c)/d_c) # Averaging Computed Delta Distance Per Subject
    
    return delta_dist

###### Iterating over Gestures
delta_distance = []
for g_id, gesture_curr in enumerate(gesture_list):
    print('==============================================')
    delta_dist_g = delta_dist_compute(Test_Embeddings,g_id,num_subjects,y_dev,y_dev_id)
    delta_distance.append(delta_dist_g)
    print('Delta Distance for '+str(gesture_curr)+' = '+str(delta_dist_g))

delta_distance = np.array(delta_distance) # Array Formation
delta_distance = (delta_distance - np.mean(delta_distance))/np.std(delta_distance) # Mean Normalization
delta_distance = delta_distance/np.linalg.norm(delta_distance) # L2-Normalization

####### Generative Capacity
###### Cosine Bounds Function
def get_cosine_bounds(X, quantile=0.05):
    cosine_dist = np.dot(X, X.transpose())
    min_val = np.min(cosine_dist, axis=1)

    mask = np.ones(cosine_dist.shape, dtype=bool)
    np.fill_diagonal(mask, 0)
    cosine_dist = cosine_dist * mask

    max_val = np.max(cosine_dist, axis=1)
    range_val = max_val - min_val

    value = np.quantile(min_val, quantile)
    total_angle = np.arccos(value) * 180 / np.pi

    return total_angle, min_val, max_val

###### Capacity Estimation Function
def ratio_hyperspherical_caps(inter_class_angle, intra_class_angle, cos_delta, sin_delta, d):
    
    # compute cos(\theta) where \theta is the solid
    # angle corresponding to the inter-class hyper-spherical cap
    cos_theta = np.cos(inter_class_angle)
    sin_theta = np.sqrt(1 - cos_theta**2)
    
    cos_omega = cos_theta * cos_delta - sin_theta * sin_delta
     
    x = 1 - cos_omega * cos_omega
    a = (d - 1)/2
    b = 0.5 

    numerator = sp.betainc(a, b, x)
    index = np.where(cos_omega < 0)
    #numerator[index] = 0.5 + numerator[index]
    
    # compute cos(\phi) where \phi is the solid
    # angle corresponding to the intra-class hyper-spherical cap
    cos_phi = np.cos(intra_class_angle)
    sin_phi = np.sqrt(1 - cos_phi**2)
    cos_omega = cos_phi * cos_delta - sin_phi * sin_delta

    x = 1 - cos_omega * cos_omega
    a = (d - 1)/2
    b = 0.5

    denominator = sp.betainc(a, b, x)
    index = np.where(cos_omega < 0)
    #denominator[index] = 0.5 + denominator[index]

    capacity = numerator / denominator
    #capacity[capacity < 1] = 1
    
    return capacity

####### Biometric Capacity
###### Defining Essentials
total_gestures = 4
total_ids = 16
g_angle = [] # List to store Intra-Gesture Angular Capacity
g_id_angle = [] # List to store Intra-Gesture Id Angular Capacity
Capacity_Value = [] # List to store Generative Biometric Capacity of each of the gesture
d_size = 32 # Embedding Size
delta = 0 # Setting Delta(FAR Parameter) to zero

###### Iteration Loop
for gesture_val in range(total_gestures): # Iterating over the Gestures

    ###### Gesture-level
    X_store = [] # List to store all the examples of that gesture
    #idx_store = [] # List to store all the indexes of the gestures being stored
    id_store = [] # List to store the identity-labels corresponding to the gesture
    g_id_angle_store = [] # List to store Angular Spreads of the 'N' identities involved in the dataset

    ##### Gesture-Store Curation
    for g_idx, X_ges in enumerate(Test_Embeddings): # Iterating over the features

        if(y_dev[g_idx] == gesture_val): # Checking for the Gesture Labels

            X_store.append(X_ges) # Storing the Feature
            id_store.append(y_dev_id[g_idx]) # Storing ID-label of the feature

    ##### Estimation of Gesture-Level Angular Spread
    g_angle_curr,_,_ = get_cosine_bounds(np.array(X_store))
    g_angle_curr = (g_angle_curr/2)*(np.pi/180)
    g_angle.append(g_angle_curr)

    ##### Estimation of Intra-Gesture Id-Level Angular Spread
    for id_idx in range(total_ids): # Searching for Particular Identities
        X_id_store = [] # List to store Intra-Gesture features of a particular identity

        for item_idx, item in enumerate(X_store): # Iterating over the Current Gesture-Store
            
            if(id_store[item_idx] == id_idx): # Identity Match-found
                X_id_store.append(item) # Storing the Feature

        #### Estimation of Intra-Gesture Intra-Id Angular Spread
        g_id_angle_curr,_,_ = get_cosine_bounds(np.array(X_id_store))
        g_id_angle_curr = (g_id_angle_curr/2)*(np.pi/180)    
        g_id_angle_store.append(g_id_angle_curr)

    g_id_angle_curr_overall = np.average(g_id_angle_store) # Avg. Angular Spread of all the Identities within the gesture under consideration
    g_id_angle.append(g_id_angle_curr_overall) # Storing in Global List

    ##### Estimation of Gesture's Biometric Capacity
    capacity_curr = ratio_hyperspherical_caps(g_angle_curr,g_id_angle_curr_overall,1,0,d_size) # Biometric Capacity of the Current Gesture
    Capacity_Value.append(capacity_curr) # Storing Values

#print(Capacity_Value)
Capacity_Value = np.array(Capacity_Value)
Capacity_Value = (Capacity_Value - np.mean(Capacity_Value))/np.std(Capacity_Value)
Capacity_Value = Capacity_Value/np.linalg.norm(Capacity_Value)

####### MasterFace Capacity
def Compute_MasterFace_Capacity(d_c_star,d_size): 

    """
    Function to Compute MasterFace based Gesture Capacity

    INPUTS:-
    1) d_c_star: Avg. Distance between different Identity Centroids within a Gesture
    2) d_size: Size of the Embeddings

    OUTPUTS:-
    1) MasterFace_Capacity: Estimated Biometric Capacity within Hand-Gestures  
    """

    MasterFace_Capacity = np.exp((d_size*(0.993 - 0.436*(1-d_c_star))+3.701-3.706*(1 - d_c_star)))
    return MasterFace_Capacity

MasterFace_Capacity = Compute_MasterFace_Capacity(np.array(d_c_star),32)
MasterFace_Capacity = (MasterFace_Capacity - np.mean(MasterFace_Capacity))/np.std(MasterFace_Capacity)
MasterFace_Capacity = MasterFace_Capacity/np.linalg.norm(MasterFace_Capacity)

##################################################################################################################
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
### Feature-Space Scores
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
##################################################################################################################

####### CGID and Decorr-CGID Score
C_I, C_D = CGID_Score_Calculator(Test_Embeddings,y_dev)
print('===============================================')
print('CGID Score: '+str(round(C_I,3)))              
print('CGID Score Decorrelated: '+str(round(C_D,3)))
print('===============================================')

##################################################################################################################
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
### Comparison Scores    
#$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$#
##################################################################################################################

####### DGBQA-Score
###### General
beta = 0.75
alpha = 2
nu = 1
rank_dev = avg_rank_deviation(e,dgbqa_score,num_gestures)
Ar = acceptance_score(dgbqa_score,e_prime,num_gestures)
d = pattern_match_dist(dgbqa_score,e_prime,num_gestures)
d_metric = (np.log2(2+nu*d)**(-1/alpha))
O_prime = np.exp(-beta*C_D) # Orthogonal Penalty
Ar_star = Ar*d_metric
Ar_star_plusplus = Ar_star*O_prime
Ar_max = acceptance_score(dgbqa_score,e_prime,num_gestures,normalizer=True)
nAr = Ar/Ar_max
nAr_star = Ar_star/Ar_max
nAr_star_plusplus = Ar_star_plusplus/Ar_max

print('Rank Deviation: '+str(rank_dev))
print('Ar: '+str(Ar))
print('d: '+str(d))
print('C_I: '+str(round(C_I,3)))
print('C_D:' +str(round(C_D,3)))
print('d_metric: '+str(d_metric))
print('O_prime: '+str(O_prime))
print('Ar_star: '+str(Ar_star))
print('Ar_star_++: '+str(Ar_star_plusplus))
print('Ar_max: '+str(Ar_max))
print('nAr: '+str(nAr))
print('nAr_star: '+str(nAr_star))
print('nAr_star_++: '+str(nAr_star_plusplus))
print('===============================================')
print(e_prime)

###### Ranking
##### DGBQA-Score
rank_dev = avg_rank_deviation(e,dgbqa_score,num_gestures)
d = pattern_match_dist(dgbqa_score,e_prime,num_gestures)
d_metric = (np.log2(2+nu*d)**(-1/alpha))
Ar_star = acceptance_score_comp(dgbqa_score,e_prime,num_gestures)*d_metric
print('======================================')
print('DGBQA-Score')
print(dgbqa_score)
print('Rank-Deviation: '+str(rank_dev)+' d: '+str(d)+' Ar_star: '+str(Ar_star))

##### d_c_star
#rank_dev = avg_rank_deviation(e,d_c_star,num_gestures)
#d = pattern_match_dist(d_c_star,e_prime,11)
#d_metric = (np.log2(2+nu*d)**(1/alpha))
#Ar_star = acceptance_score_comp(d_c_star,e_prime,11)*d_metric
#print('======================================')
#print('d_c_star')
#print('Rank-Deviation: '+str(rank_dev)+' Ar_star: '+str(Ar_star))

##### d_cs
#rank_dev = avg_rank_deviation(e,d_cs,num_gestures)
#d = pattern_match_dist(d_cs,e_prime,11)
#d_metric = (np.log2(2+nu*d)**(1/alpha))
#Ar_star = acceptance_score_comp(d_cs,e_prime,11)*d_metric
#print('======================================')
#print('d_cs')
#print('Rank-Deviation: '+str(rank_dev)+' Ar_star: '+str(Ar_star))

##### DGBQA-Score W/o Penalty
rank_dev = avg_rank_deviation(e,dgbqa_score_wo,num_gestures)
d = pattern_match_dist(dgbqa_score_wo,e_prime,num_gestures)
d_metric = (np.log2(2+nu*d)**(-1/alpha))
Ar_star = acceptance_score_comp(dgbqa_score_wo,e_prime,num_gestures)*d_metric
print('======================================')
print('DGBQA-Score W/o Penalty')
print(dgbqa_score_wo)
print('Rank-Deviation: '+str(rank_dev)+' d: '+str(d)+' Ar_star: '+str(Ar_star))

##### Delta Distance
rank_dev = avg_rank_deviation(e,delta_distance,num_gestures)
d = pattern_match_dist(delta_distance,e_prime,num_gestures)
d_metric = (np.log2(2+nu*d)**(-1/alpha))
Ar_star = acceptance_score_comp(delta_distance,e_prime,num_gestures)*d_metric
print('======================================')
print('delta_distance')
print(delta_distance)
print('Rank-Deviation: '+str(rank_dev)+' d: '+str(d)+' Ar_star: '+str(Ar_star))

##### Generative Capacity
rank_dev = avg_rank_deviation(e,Capacity_Value,num_gestures)
d = pattern_match_dist(Capacity_Value,e_prime,num_gestures)
d_metric = (np.log2(2+nu*d)**(-1/alpha))
Ar_star = acceptance_score_comp(Capacity_Value,e_prime,num_gestures)*d_metric
print('======================================')
print('Generative Capacity')
print(Capacity_Value)
print('Rank-Deviation: '+str(rank_dev)+' d: '+str(d)+' Ar_star: '+str(Ar_star))

##### MasterFace Capacity
rank_dev = avg_rank_deviation(e,MasterFace_Capacity,num_gestures)
d = pattern_match_dist(MasterFace_Capacity,e_prime,num_gestures)
d_metric = (np.log2(2+nu*d)**(-1/alpha))
Ar_star = acceptance_score_comp(MasterFace_Capacity,e_prime,num_gestures)*d_metric
print('======================================')
print('MasterFace')
print(MasterFace_Capacity)
print('Rank-Deviation: '+str(rank_dev)+' d: '+str(d)+' Ar_star: '+str(Ar_star))

Inter-Distance - d_c_star: 0.5013526347776254
Intra-Distance - d_cs_avg: 0.6298872213810682
Inter-Distance - d_c_star: 0.6765149756024281
Intra-Distance - d_cs_avg: 0.6615247540175915
Inter-Distance - d_c_star: 0.7872827542324861
Intra-Distance - d_cs_avg: 0.6384690115228295
Inter-Distance - d_c_star: 0.9045808727542559
Intra-Distance - d_cs_avg: 0.6362616270780563
[-0.3769924610633828, 0.037261142540255365, 0.34947880259402064, 0.6043872990155489]
Average Intra-Subject Separation: 0.6298872213810682
Minimum Intra-Subject Separation: 0.36934810876846313
Maximum Intra-Subject Separation (d_cs): 0.8029829263687134
Delta Distance for Compass = 0.21556585987503693
Average Intra-Subject Separation: 0.6615247540175915
Minimum Intra-Subject Separation: 0.2828809916973114
Maximum Intra-Subject Separation (d_cs): 1.0082943439483643
Delta Distance for Piano = 0.3439170238453021
Average Intra-Subject Separation: 0.6384690115228295
Minimum Intra-Subject Separation: 0.24417833983898163
Maximum Intr

C:\Users\ASUS\AppData\Local\Temp\ipykernel_22400\784039247.py:180: DeprecationWarning: Calling nonzero on 0d arrays is deprecated, as it behaves surprisingly. Use `atleast_1d(cond).nonzero()` if the old behavior was intended. If the context of this warning is of the form `arr[nonzero(cond)]`, just use `arr[cond]`.
  index = np.where(cos_omega < 0)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_22400\784039247.py:194: DeprecationWarning: Calling nonzero on 0d arrays is deprecated, as it behaves surprisingly. Use `atleast_1d(cond).nonzero()` if the old behavior was intended. If the context of this warning is of the form `arr[nonzero(cond)]`, just use `arr[cond]`.
  index = np.where(cos_omega < 0)


CGID Score: 0.255
CGID Score Decorrelated: 0.255
Rank Deviation: 1.5
Ar: 4.865848837237573
d: 5.16473134415489
C_I: 0.255
C_D:0.255
d_metric: 0.5932955116313361
O_prime: 0.8258069642622554
Ar_star: 2.8868862754096076
Ar_star_++: 2.3840107912663773
Ar_max: 17.00722685148489
nAr: 0.28610477650051097
nAr_star: 0.1697446797540397
nAr_star_++: 0.14017633868735224
[ 0.42361379  0.17991043 -0.85224501  0.24872079]
DGBQA-Score
[-0.72421076 -0.15872136  0.26748079  0.61545133]
Rank-Deviation: 1.5 d: 5.16473134415489 Ar_star: 0.9213889932093712
DGBQA-Score W/o Penalty
[-0.66044547 -0.23622259  0.21811254  0.67855552]
Rank-Deviation: 1.5 d: 5.346544582940867 Ar_star: 0.9155816004492466
delta_distance
[-0.7247194  -0.07246791  0.67407177  0.12311554]
Rank-Deviation: 1.5 d: 4.328907698119722 Ar_star: 1.2868583206889652
Generative Capacity
[-0.52875425 -0.46892243  0.46108035  0.53659633]
Rank-Deviation: 1.5 d: 5.7403110769521355 Ar_star: 0.9038263122044703
MasterFace
[-0.28903674 -0.28903524 -0.287

# Per-Gesture Analysis

In [ ]:
###### tSNE Embeddings
tsne_embeddings = TSNE(n_components=2,perplexity=30,learning_rate=10,n_iter=10000,n_iter_without_progress=50).fit_transform(Test_Embeddings)

####### Iterating over Classes
plt.rcParams["figure.figsize"] = [12,10]
for g_idx,g_label in zip(np.arange(4),['Compass','Piano','Push','Flipping Fist']):
    
    print('=======================================================================================================================')
    print('G'+str(g_idx)+' : '+str(g_label))
    
    tsne_embeddings_c = tsne_embeddings[y_dev == g_idx] # tsne embeddings
    y_dev_id_c = y_dev_id[y_dev == g_idx] # Identity Labels
    
    for s_idx,color_index in zip(list(np.arange(16)),mcp.gen_color(cmap="nipy_spectral",n=16)):
        plt.scatter(tsne_embeddings_c[y_dev_id_c == s_idx, 0],tsne_embeddings_c[y_dev_id_c == s_idx, 1],s=400,color=color_index,edgecolors='k',marker='h')
        plt.grid(axis='both')
        plt.legend(['s1','s2','s3','s4','s5','s6','s7','s8','s9','s10','s11','s12','s13','s14','s15','s16'],
                   loc='upper right',prop={'size': 12})
        plt.title('Res3D-ViViT: '+str(g_label))
        plt.grid(b='True',which='both')
    plt.show()